# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene

from utilis import *

# Load Data

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head()

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

In [ ]:
(df['p-tau217'] > 0.444).value_counts()
# (df[df['Gender'] == 'f']['p-tau217'] > 0.444).value_counts()

# Settings

In [ ]:
# Graphs Settings
healthy_color = '#b4cde3' # blue
risk_color = '#fbb4ae' # red
inter_color = '#fed9a6'
pie_colors = [healthy_color, inter_color, risk_color] # blue, orange, red
vline_color = 'red'

In [ ]:
TAU_217 = 'p-tau217'
TAU_CAT = 'tau_217_category'
HIGH_TAU_217_THR = 0.53
INT_TAU_217_THR = 0.445

TMT_B_COL = 'Trail_B_(second)'
MBI_C_COL = 'Mild_behavioral_Impairment_Checklist'
IST_COL = 'IST'
ANEXITY_COL = 'Anxiety/depression'

MCI_THR = 26
MOCA_COL = 'MOCA_score'
MIN_MOCA_SCORE = 19

## Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
outliers = []  # Participant exclusions are maintained privately.

print(df.shape) 
df = df.drop(df[df['ID'].isin(outliers)].index)
print(df.shape) 

## Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
# removing samples with moca below the minimum threshold of 19
df = df[df[MOCA_COL] > MIN_MOCA_SCORE]
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])

In [ ]:
df[TAU_CAT].value_counts()

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

In [ ]:
sns.histplot(data=df, x=TAU_217, kde=True, stat='count', element='step')

plt.legend()
plt.title('Histogram of tau217 levels')
plt.savefig(SAVE_DIR + 'tau217_hist.pdf')

In [ ]:
sns.kdeplot(data=df, x=TAU_217, alpha=0.5, fill=True, clip=(df[TAU_217].min(), df[TAU_217].max()))
plt.title('Tau217 Density Plot')
plt.legend()

plt.savefig(SAVE_DIR + 'tau217_hist.pdf')

Cognitive screening and defining cognitive un-impaired (MoCA.26) and MCI (MoCA 19-26)

In [ ]:
sns.histplot(data=df.dropna(subset=MOCA_COL), x=MOCA_COL, kde=True, stat='count', element='step', bins=10)
plt.vlines(MCI_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='MCI threshold', linestyles='dashed')

plt.legend(loc='upper left')
plt.title('Histogram of MoCA levels')
plt.savefig(SAVE_DIR + 'MOCA_hist.pdf')


In [ ]:
sns.kdeplot(data=df, x=MOCA_COL, alpha=0.5, fill=True, clip=(df[MOCA_COL].min(), df[MOCA_COL].max()))
plt.vlines(MCI_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='MCI threshold', linestyles='dashed')

plt.legend(loc='upper left')
plt.title('Density plot of MoCA levels')
plt.savefig(SAVE_DIR + 'MOCA_Density_plot.pdf')


# 2. Characteristics of the (pre-clinical) cohort

Fraction of participants with cognitive decline in spite of self-reported cognitive un-impairment (24% of self reportedly cognitive intact participants displayed MoCA score of MCI)

#### Full data

In [ ]:
df_full['MCI'].value_counts()

In [ ]:
healthy_n = (df_full['MCI'] == False).sum()
mci_n = (df_full['MCI'] == True).sum()
explode = (0.1, 0.1)
graph_colors = [healthy_color, risk_color]

plt.pie(df_full['MCI'].value_counts(), labels=['Cognitive\nHealthy', 'Cognitive\nImpaired'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
plt.title('Cognitive healthy vs Impaired distribution')
plt.savefig(SAVE_DIR + 'Cognitive_healthy_vs_Impaired_pie.pdf')

#### Only tau-217 samples

In [ ]:
healthy_n = (df['MCI'] == False).sum()
mci_n = (df['MCI'] == True).sum()
explode = (0.1, 0.1)
graph_colors = [healthy_color, risk_color]
plt.pie(df['MCI'].value_counts(), labels=['Cognitive\nHealthy', 'Cognitive\nImpaired'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
# plt.legend(labels=[f'Cognitive Healthy N={healthy_n}', f'Cognitive Impaired N={mci_n}'], loc='upper right')
plt.title('Cognitive healthy vs Impaired distribution')
# plt.savefig(SAVE_DIR + 'Cognitive_healthy_vs_Impaired_pie.pdf')

In [ ]:
explode = (0.1, 0.1, 0.1)
low_tau_n, intr_tau_n, high_tau_n = (df[TAU_CAT]=='Low').sum(),  (df[TAU_CAT]=='Intermediate').sum(), (df[TAU_CAT]=='High').sum()
graph_colors = [healthy_color, inter_color, risk_color]

plt.pie(df[TAU_CAT].value_counts(),
         labels=['Low Tau', 'Intermedaite Tau', 'High Tau'],
           autopct='%1.1f%%', colors=graph_colors,
             shadow=True, explode=explode)
# plt.legend(labels=[f'Low Tau N={low_tau_n}', f'Intermedaite-Low Tau N={intr_low_tau_n}', f'Intermedaite-High Tau N={intr_high_tau_n}', f'High Tau N={high_tau_n}'], loc='upper right')
# plt.legend(loc='upper right')
plt.title('Tau 217 Discrete distribution')
plt.savefig(SAVE_DIR + 'Tau_217_Discrete_distribution_pie.pdf')

Age Disterbuition

In [ ]:
sns.histplot(data=df, x='Age', kde=True, stat='count', element='step')

plt.title('Histogram of Age Disterbuition')
plt.savefig(SAVE_DIR + 'Age_hist.pdf')


In [ ]:
sns.kdeplot(data=df['Age'], alpha=0.5, fill=True, clip=(df['Age'].min(), df['Age'].max()))
plt.title('Age Density Plot')

plt.savefig(SAVE_DIR + 'Age_density.pdf')


Sex Distribution

In [ ]:
df['Gender'].value_counts()

In [ ]:
females_n = (df['Gender'] == 'f').sum()
males_n = (df['Gender'] == 'm').sum()
explode = (0.05, 0.05)
graph_colors = [risk_color, healthy_color]
plt.pie(df['Gender'].value_counts(), labels=['Females', 'Males'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode, startangle=90)
# plt.legend(labels=[f'Cognitive Healthy N={healthy_n}', f'Cognitive Impaired N={mci_n}'], loc='upper right')
plt.title('Sex distribution')
plt.savefig(SAVE_DIR + 'Sex_pie.pdf')

# 3.	Comparison between p-tau217 high- versus low- participants

Performance in other cognitive (TMT-B, IST) and behavioral (MBI-C) tests in pre-clinical AD versus control groups (pre-clinical AD exhibit lower performance in TMT-B and IST tests and higher anxiety, indicating cognitive decline already at the pre-clinical stage]

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

## Cognitive tests

In [ ]:
COGNITIVE_SAVE_DIR = SAVE_DIR + 'cognitive/'

Filter cognitive damaged subjects

In [ ]:
df_high_vs_low_no_mci = df_high_vs_low[df_high_vs_low['MCI'] == False]
df_high_vs_low_no_mci[TAU_CAT].value_counts()

### Executive

In [ ]:
EXECTUTIVE = 'Executive'

df_high_vs_low_exe = df_high_vs_low_no_mci[df_high_vs_low_no_mci[EXECTUTIVE].notna()]
graph_colors = [healthy_color, risk_color]
kdeplot_high_vs_low(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, graph_colors, 'Executive vs Tau 217 category',COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, 4, graph_colors, 'Executive vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'Low'][EXECTUTIVE]
group2 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'High'][EXECTUTIVE]

stat_test(data_type, group1, group2)

### Spatial

In [ ]:
SPATIAL = 'Spatial'
df_high_vs_low_spatial = df_high_vs_low_no_mci[df_high_vs_low_no_mci[SPATIAL].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_spatial, SPATIAL, TAU_CAT, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_spatial, SPATIAL, TAU_CAT, 4, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'Low'][SPATIAL]
group2 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'High'][SPATIAL]

stat_test(data_type, group1, group2)

### Attention_and_concentration

In [ ]:
A_C = 'Attention_and_concentration'
df_high_vs_low_ac = df_high_vs_low_no_mci[df_high_vs_low_no_mci[A_C].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ac, A_C, TAU_CAT, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_ac, A_C, TAU_CAT, 5, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Ordinal'
group1 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'Low'][A_C]
group2 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'High'][A_C]

stat_test(data_type, group1, group2)

### Memory

In [ ]:
MEMORY = 'Memory'
df_high_vs_low_memory = df_high_vs_low_no_mci[df_high_vs_low_no_mci[MEMORY].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_memory, MEMORY, TAU_CAT, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_memory, MEMORY, TAU_CAT, 5, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'Low'][MEMORY]
group2 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'High'][MEMORY]

stat_test(data_type, group1, group2)

### Language

In [ ]:
LANGUAGE = 'Language'
df_high_vs_low_language = df_high_vs_low_no_mci[df_high_vs_low_no_mci[LANGUAGE].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_language, LANGUAGE, TAU_CAT, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_language, LANGUAGE, TAU_CAT, 5, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_hist.pdf')

In [ ]:
# statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'Low'][LANGUAGE]
group2 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'High'][LANGUAGE]

stat_test(data_type, group1, group2)

### Orientation

In [ ]:
ORIENTATION = 'Orientation'

df_high_vs_low_orientation = df_high_vs_low_no_mci[df_high_vs_low_no_mci[ORIENTATION].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, 6, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_orientation
group2 = [TAU_CAT, ORIENTATION]

stat_test(data_type, group1, group2) # Too few samples for the test

### Frontal memory score (#1)

In [ ]:
FMC = 'Frontal memory score (#1)'

df_high_vs_low_fmc = df_high_vs_low_no_mci[df_high_vs_low_no_mci[FMC].notna()]
df_high_vs_low_fmc[FMC] = df_high_vs_low_fmc[FMC].astype(float)
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_fmc, FMC, TAU_CAT, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_fmc, FMC, TAU_CAT, 4, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'Low'][FMC]
group2 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'High'][FMC]

stat_test(data_type, group1, group2)

### Hipocampal memory score (#2)

In [ ]:
HMC = 'Hipocampal memory score (#2)'

df_high_vs_low_hmc = df_high_vs_low_no_mci[df_high_vs_low_no_mci[HMC].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_hmc, HMC, TAU_CAT, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_hmc, HMC, TAU_CAT, 5, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'Low'][HMC]
group2 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'High'][HMC]

stat_test(data_type, group1, group2)

## behavioral tests

In [ ]:
BEHAVIORAL_SAVE_DIR = SAVE_DIR + 'behavioral/'

### TMT (Trail B)

In [ ]:
df_high_vs_low_tmt = df_high_vs_low.dropna(subset=[TMT_B_COL])
df_high_vs_low_tmt[TMT_B_COL] = df_high_vs_low_tmt[TMT_B_COL].astype(float)
graph_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_tmt, TMT_B_COL, TAU_CAT, graph_colors, 'Trail B vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Trail_B_vs_Tau_217_category_kde.pdf')

In [ ]:
sns.violinplot(data=df_high_vs_low, x=TAU_CAT, y=TMT_B_COL, palette=graph_colors, inner='quartile')
plt.title('Trail B vs Tau 217 category')

In [ ]:
# Statistical test
data_type = 'Continuous'
group1 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'Low'][TMT_B_COL]
group2 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'High'][TMT_B_COL]

stat_test(data_type, group1, group2)

### MBI-C

In [ ]:
df_high_vs_low_mbi = df_high_vs_low.dropna(subset=[MBI_C_COL])
df_high_vs_low_mbi[MBI_C_COL] = df_high_vs_low_mbi[MBI_C_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_mbi, MBI_C_COL, TAU_CAT, graph_colors, 'MBI-C vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'MBI_C_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
sns.violinplot(data=df_high_vs_low_mbi, x=TAU_CAT, y=MBI_C_COL, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('MBI-C vs Tau 217 category')

plt.savefig(SAVE_DIR + 'MBI_C_vs_Tau_217_category_violin.pdf')

In [ ]:
# threshold MBI-C
MBI_C_THR = 6
df_high_vs_low_mbi['MBI-C_THR'] = (df_high_vs_low_mbi[MBI_C_COL] > MBI_C_THR)
# df_high_vs_low_mbi_c_thr['MBI-C_THR']

df_high_vs_low_mbi_c_thr = df_high_vs_low_mbi.dropna(subset=['MBI-C_THR'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_mbi_c_thr, 'MBI-C_THR', TAU_CAT, 2 , graph_colors, 'MBI-C_THR vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Anxiety_Depression_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Ordinal'

group1 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'Low'][MBI_C_COL]
group2 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'High'][MBI_C_COL]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_mbi
group2 = [TAU_CAT, 'MBI-C_THR']

stat_test(data_type, group1, group2) # Too few samples for the test

### IST

In [ ]:
df_high_vs_low_ist = df_high_vs_low.dropna(subset=[IST_COL])
df_high_vs_low_ist[IST_COL] = df_high_vs_low_ist[IST_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_ist, IST_COL, TAU_CAT, graph_colors, 'IST vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'IST_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
sns.violinplot(data=df_high_vs_low_ist, x=TAU_CAT, y=IST_COL, hue=TAU_CAT, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('IST vs Tau 217 category')

plt.savefig(BEHAVIORAL_SAVE_DIR + 'IST_vs_Tau_217_category_violin.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'Low'][IST_COL]
group2 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'High'][IST_COL]

stat_test(data_type, group1, group2)

### interest, motivation, and drive

In [ ]:
IMD = 'interest, motivation, and drive'

df_high_vs_low_imd = df_high_vs_low.dropna(subset=[IMD])
df_high_vs_low_imd[IMD] = df_high_vs_low_imd[IMD].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_imd, IMD, TAU_CAT, graph_colors, 'Interest, Motivation, and Drive vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Interest_Motivation_and_Drive_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# categorical data
IMD_THR = 2

df_high_vs_low_imd['IMD_THR'] = (df_high_vs_low_imd[IMD] > IMD_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_imd, 'IMD_THR', TAU_CAT, 2 , graph_colors, 'Interest, Motivation, and Drive vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Interest_Motivation_and_Drive_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_imd[df_high_vs_low_imd[TAU_CAT] == 'Low'][IMD]
group2 = df_high_vs_low_imd[df_high_vs_low_imd[TAU_CAT] == 'High'][IMD]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_imd
group2 = [TAU_CAT, 'IMD_THR']

stat_test(data_type, group1, group2) # Too few samples for the test

### Impulse control 

In [ ]:
IC = 'Impulse control'

df_high_vs_low_ic = df_high_vs_low.dropna(subset=[IC])
df_high_vs_low_ic[IC] = df_high_vs_low_ic[IC].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ic, IC, TAU_CAT, graph_colors, 'Impulse Control vs Tau 217 category',BEHAVIORAL_SAVE_DIR + 'Impulse_Control_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Impulse control threshold
IC_THR = 2
df_high_vs_low_ic['IC_THR'] = (df_high_vs_low_ic[IC] > IC_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ic, 'IC_THR', TAU_CAT, 2 , graph_colors, 'Impulse Control vs Tau 217 category', BEHAVIORAL_SAVE_DIR + 'Impulse_Control_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ic[df_high_vs_low_ic[TAU_CAT] == 'Low'][IC]
group2 = df_high_vs_low_ic[df_high_vs_low_ic[TAU_CAT] == 'High'][IC]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_ic
group2 = [TAU_CAT, 'IC_THR']

stat_test(data_type, group1, group2) # Too few samples for the test

### following societal norms

In [ ]:
SOCIETAL_NORMS = 'following societal norms'

df_high_vs_low_fcn = df_high_vs_low.dropna(subset=[SOCIETAL_NORMS])
df_high_vs_low_fcn[SOCIETAL_NORMS] = df_high_vs_low_fcn[SOCIETAL_NORMS].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_fcn, SOCIETAL_NORMS, TAU_CAT, graph_colors, 'following societal norms',BEHAVIORAL_SAVE_DIR + 'following societal norms_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# categorical data
FCN_THR = 0
df_high_vs_low_fcn['FCN_THR'] = (df_high_vs_low_fcn[SOCIETAL_NORMS] > FCN_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_fcn, 'FCN_THR', TAU_CAT, 2 , graph_colors, 'following societal norms', BEHAVIORAL_SAVE_DIR + 'following societal norms_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_fcn[df_high_vs_low_fcn[TAU_CAT] == 'Low'][SOCIETAL_NORMS]
group2 = df_high_vs_low_fcn[df_high_vs_low_fcn[TAU_CAT] == 'High'][SOCIETAL_NORMS]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_fcn
group2 = [TAU_CAT, 'FCN_THR']

stat_test(data_type, group1, group2) # Too few samples for the test

### strongly held beliefs and sensory

In [ ]:
STRONGLY_HELD_BELIEFS = 'strongly held beliefs and sensory'

df_high_vs_low_shb = df_high_vs_low.dropna(subset=[STRONGLY_HELD_BELIEFS])
df_high_vs_low_shb[STRONGLY_HELD_BELIEFS] = df_high_vs_low_shb[STRONGLY_HELD_BELIEFS].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_shb, STRONGLY_HELD_BELIEFS, TAU_CAT, graph_colors, 'strongly held beliefs and sensory',BEHAVIORAL_SAVE_DIR + 'strongly held beliefs and sensory_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Categorical data
SHB_THR = 0
df_high_vs_low_shb['SHB_THR'] = (df_high_vs_low_shb[STRONGLY_HELD_BELIEFS] > SHB_THR)
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_shb, 'SHB_THR', TAU_CAT, 2 , graph_colors, 'strongly held beliefs and sensory', BEHAVIORAL_SAVE_DIR + 'strongly held beliefs and sensory_categorize_vs_Tau_217_category_hist.pdf')

### iADL

In [ ]:
IADL = 'iADL'

df_high_vs_low_iadl = df_high_vs_low.dropna(subset=[IADL])
df_high_vs_low_iadl[IADL] = df_high_vs_low_iadl[IADL].astype(float)
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_iadl, IADL, TAU_CAT, graph_colors, 'iADL',BEHAVIORAL_SAVE_DIR + 'iADL_vs_Tau_217_category_kde.pdf', alpha=0.5)

## Risk Factors

In [ ]:
RISK_FACTOR_DIR = SAVE_DIR + 'risk_factors/'

### Total number of risk factors (not increased in pre-clinical AD as compared to low p-tau matched controls).

In [ ]:
N_RISK_COL = 'Number_of_risk_factors'
df_high_vs_low_risk = df_high_vs_low.dropna(subset=[N_RISK_COL])
df_high_vs_low_risk[N_RISK_COL] = df_high_vs_low_risk[N_RISK_COL].astype(int)
grpah_colors = [healthy_color, risk_color]

In [ ]:
# stacked histogram of number of risk factors
sns.histplot(data=df_high_vs_low_risk, x=N_RISK_COL, hue=TAU_CAT, multiple='stack', bins=9, palette=grpah_colors, kde=True)
plt.legend(labels=['High Tau', 'Low Tau'])

plt.title('Number of risk factors vs Tau 217 category')
plt.savefig(RISK_FACTOR_DIR + 'Number_of_risk_factors_distribution.pdf')


In [ ]:
kdeplot_high_vs_low(df_high_vs_low_risk, N_RISK_COL, TAU_CAT, graph_colors, 'Number of risk factors vs Tau 217 category',RISK_FACTOR_DIR + 'Number_of_risk_factors_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_risk[df_high_vs_low_risk[TAU_CAT] == 'Low'][N_RISK_COL]
group2 = df_high_vs_low_risk[df_high_vs_low_risk[TAU_CAT] == 'High'][N_RISK_COL]

stat_test(data_type, group1, group2)

### Anexity and Depression

In [ ]:
df_high_vs_low_anx = df_high_vs_low.dropna(subset=[ANEXITY_COL])
grpah_colors = [healthy_color, risk_color] * 2

plot_histplot(df_high_vs_low_anx, ANEXITY_COL, TAU_CAT, 2 , graph_colors, 'Anxiety/Depression vs Tau 217 category', RISK_FACTOR_DIR + 'Anxiety_Depression_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_anx
group2 = [TAU_CAT, ANEXITY_COL]

stat_test(data_type, group1, group2) # Too few samples for the test

### BMI

In [ ]:
BMI = 'BMI'

df_high_vs_low_bmi = df_high_vs_low.dropna(subset=[BMI])
df_high_vs_low_bmi = df[pd.to_numeric(df[BMI], errors='coerce').notna()]
df_high_vs_low_bmi[BMI] = df_high_vs_low_bmi[BMI].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
b = ((df_high_vs_low[TAU_CAT] == 'Low')).sum()
a = ((df_high_vs_low[TAU_CAT] == 'High')).sum()
a, b

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_bmi, BMI, TAU_CAT, graph_colors, 'BMI vs Tau 217 category', RISK_FACTOR_DIR + 'BMI_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_bmi[df_high_vs_low_bmi[TAU_CAT] == 'Low'][BMI]
group2 = df_high_vs_low_bmi[df_high_vs_low_bmi[TAU_CAT] == 'High'][BMI]

stat_test(data_type, group1, group2)

###   Heart disease

In [ ]:
df['Heart_disease'].value_counts()

In [ ]:
HD = 'Heart_disease'

df_high_vs_low_hd = df_high_vs_low.dropna(subset=[HD])
grpah_colors = [healthy_color, risk_color]

In [ ]:
plot_histplot(df_high_vs_low_hd, HD, TAU_CAT, 2, graph_colors, 'Heart disease vs Tau 217 category', RISK_FACTOR_DIR + 'Heart_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hd
group2 = [TAU_CAT, HD]

stat_test(data_type, group1, group2) # Too few samples for the test

### Diabetes

In [ ]:
DIABETES = 'Diabetes'
df_high_vs_low_d = df_high_vs_low.dropna(subset=[DIABETES])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_d, DIABETES, TAU_CAT, 2 , graph_colors, 'Diabetes vs Tau 217 category', RISK_FACTOR_DIR + 'Diabetes_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_d
group2 = [TAU_CAT, DIABETES]

stat_test(data_type, group1, group2) # Too few samples for the test

###  Hypertension    	

In [ ]:
HYP = 'Hypertension'

df_high_vs_low_hyp = df_high_vs_low.dropna(subset=[HYP])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_d, HYP, TAU_CAT, 2, grpah_colors, 'Hypertension vs Tau 217 category', RISK_FACTOR_DIR + 'Hypertension_vs_Tau_217_category_violin.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_hyp
group2 = [TAU_CAT, HYP]

stat_test(data_type, group1, group2) # Too few samples for the test

### Hyperlipidemia

In [ ]:
HL = 'Hyperlipidemia'

df_high_vs_low_hl = df_high_vs_low.dropna(subset=[HL])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_hl, HL, TAU_CAT, 2, grpah_colors, 'Hyperlipidemia vs Tau 217 category', RISK_FACTOR_DIR + 'Hyperlipidemia_vs_Tau_217_category_hist.pdf') 

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hl
group2 = [TAU_CAT, HL]

stat_test(data_type, group1, group2)

### Periodontal disease

In [ ]:
PERI = 'Periodontal disease'

df_high_vs_low_peri = df_high_vs_low.dropna(subset=[PERI])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_peri, PERI, TAU_CAT, 2, grpah_colors, 'Periodontal disease vs Tau 217 category', RISK_FACTOR_DIR + 'Periodontal_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_peri
group2 = [TAU_CAT, PERI]

stat_test(data_type, group1, group2) # Too few samples for the test

### Recurrent_UTI/_pneumonia/other_infection

In [ ]:
UTI = 'Recurrent_UTI/_pneumonia/other_infection'

df_high_vs_low_uti = df_high_vs_low.dropna(subset=[UTI])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_uti, UTI, TAU_CAT, 2, grpah_colors, 'Recurrent UTI/pneumonia/other infection vs Tau 217 category', RISK_FACTOR_DIR + 'Recurrent_UTI_pneumonia_other_infection_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_uti
group2 = [TAU_CAT, UTI]

stat_test(data_type, group1, group2) # Too few samples for the test

### Systemic_inflammatory_disease

In [ ]:
SID = 'Systemic_inflammatory_disease'

df_high_vs_low_sid = df_high_vs_low.dropna(subset=[SID])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_sid, SID, TAU_CAT, 2, grpah_colors, 'Systemic inflammatory disease vs Tau 217 category', RISK_FACTOR_DIR + 'Systemic_inflammatory_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_sid
group2 = [TAU_CAT, SID]

stat_test(data_type, group1, group2) # Too few samples for the test

### Smoking 


In [ ]:
SMOKING = 'Smoking'

df_high_vs_low_smoking = df_high_vs_low.dropna(subset=[SMOKING])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_smoking, SMOKING, TAU_CAT, 2, grpah_colors, 'Smoking vs Tau 217 category', RISK_FACTOR_DIR + 'Smoking_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_smoking[df_high_vs_low_smoking[TAU_CAT] == 'Low'][SMOKING].replace({'NO': 0, 'in_the_past': 1, 'YES': 2})
group2 = df_high_vs_low_smoking[df_high_vs_low_smoking[TAU_CAT] == 'High'][SMOKING].replace({'NO': 0, 'in_the_past': 1, 'YES': 2})

stat_test(data_type, group1, group2)

### Pack_years 

In [ ]:
PACK_YEARS = 'Pack_years'

df_high_vs_low_pack_years = df_high_vs_low.dropna(subset=[PACK_YEARS])
df_high_vs_low_pack_years = df_high_vs_low_pack_years[df_high_vs_low_pack_years[PACK_YEARS] > 0]
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_pack_years, PACK_YEARS, TAU_CAT, graph_colors, 'Pack years vs Tau 217 category', RISK_FACTOR_DIR + 'Pack_years_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_pack_years[df_high_vs_low_pack_years[TAU_CAT] == 'Low'][PACK_YEARS]
group2 = df_high_vs_low_pack_years[df_high_vs_low_pack_years[TAU_CAT] == 'High'][PACK_YEARS]

stat_test(data_type, group1, group2)

### Alcohol_consumption

In [ ]:
ALCOHOL = 'Alcohol_consumption'

df_high_vs_low_alcohol = df_high_vs_low.dropna(subset=[ALCOHOL])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_alcohol, ALCOHOL, TAU_CAT, 2, grpah_colors, 'Alcohol consumption vs Tau 217 category', RISK_FACTOR_DIR + 'Alcohol_consumption_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_alcohol
group2 = [TAU_CAT, ALCOHOL]

stat_test(data_type, group1, group2) # Too few samples for the test

###  Insomnia

In [ ]:
INSOMNIA = 'Insomnia'

df_high_vs_low_insomnia = df_high_vs_low.dropna(subset=[INSOMNIA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_insomnia, INSOMNIA, TAU_CAT, 2, grpah_colors, 'Insomnia vs Tau 217 category', RISK_FACTOR_DIR + 'Insomnia_vs_Tau_217_category_hist.pdf')

In [ ]:
# STATISTICAL TEST
data_type = 'Categorical'

group1 = df_high_vs_low_insomnia[df_high_vs_low_insomnia[INSOMNIA] != 'in_the_past'] # Only one subject has 'in_the_past' value
group2 = [TAU_CAT, INSOMNIA]

stat_test(data_type, group1, group2)

### OSA 

In [ ]:
OSA = 'OSA'

df_high_vs_low_osa = df_high_vs_low.dropna(subset=[OSA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_osa, OSA, TAU_CAT, 2, grpah_colors, 'OSA vs Tau 217 category', RISK_FACTOR_DIR + 'OSA_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_osa
group2 = [TAU_CAT, OSA]

stat_test(data_type, group1, group2)

### severity of OSA

In [ ]:
OSA_SEVERITY = 'severity of OSA'

df_high_vs_low_osa_severity = df_high_vs_low.dropna(subset=[OSA_SEVERITY])
df_high_vs_low_osa_severity = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[OSA_SEVERITY] > 0]
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_osa_severity, OSA_SEVERITY, TAU_CAT, 3, grpah_colors, 'OSA severity vs Tau 217 category', RISK_FACTOR_DIR + 'OSA_severity_vs_Tau_217_category_hist.pdf')

In [ ]:
#Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[TAU_CAT] == 'Low'][OSA_SEVERITY]
group2 = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[TAU_CAT] == 'High'][OSA_SEVERITY]

stat_test(data_type, group1, group2)

### Physical activity

In [ ]:
PA = 'Physical activity'

df_high_vs_low[PA].replace({'LOW ': 'LOW', 'HIGH ': 'HIGH', 'MEDIUM ': 'MEDIUM'}, inplace=True)

In [ ]:
PA = 'Physical activity'

df_high_vs_low_pa = df_high_vs_low.dropna(subset=[PA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_pa, PA, TAU_CAT, 2, grpah_colors, 'Physical activity vs Tau 217 category', RISK_FACTOR_DIR + 'Physical_activity_vs_Tau_217_category_hist.pdf')

In [ ]:
sns.histplot(df_high_vs_low_pa, x=PA, hue=TAU_CAT, multiple='stack', bins=3, palette=grpah_colors, kde=True, fill=True)
plt.title('Physical activity vs Tau 217 category')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_pa[df_high_vs_low_pa[TAU_CAT] == 'Low'][PA].replace({'NO': 0, 'LOW': 1, 'MEDIUM': 2, 'HIGH': 3})
group2 = df_high_vs_low_pa[df_high_vs_low_pa[TAU_CAT] == 'High'][PA].replace({'NO': 0, 'LOW': 1, 'MEDIUM': 2, 'HIGH': 3})

stat_test(data_type, group1, group2)

### Time_Physical_activity

In [ ]:
TIME_PA = 'Time_Physical_activity_(hours/week)'

df_high_vs_low_time_pa = df_high_vs_low.dropna(subset=[TIME_PA])
grpah_colors = [healthy_color, risk_color]
kdeplot_high_vs_low(df_high_vs_low_time_pa, TIME_PA, TAU_CAT, graph_colors, 'Time Physical activity vs Tau 217 category', RISK_FACTOR_DIR + 'Time_Physical_activity_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_time_pa[df_high_vs_low_time_pa[TAU_CAT] == 'Low'][TIME_PA]
group2 = df_high_vs_low_time_pa[df_high_vs_low_time_pa[TAU_CAT] == 'High'][TIME_PA]

stat_test(data_type, group1, group2)

### Cognitive_activity 

In [ ]:
CA = 'Cognitive_activity'

df_high_vs_low_ca = df_high_vs_low.dropna(subset=[CA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ca, CA, TAU_CAT, 2, grpah_colors, 'Cognitive activity vs Tau 217 category', RISK_FACTOR_DIR + 'Cognitive_activity_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_ca
group2 = [TAU_CAT, CA]

stat_test(data_type, group1, group2) # Too few samples for the test

### AD_in_the_family

In [ ]:
AD_IF = 'AD_in_the_family'

df_high_vs_low_ad_ip = df_high_vs_low.dropna(subset=[AD_IF])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ad_ip, AD_IF, TAU_CAT, 2, grpah_colors, 'AD in the family vs Tau 217 category', RISK_FACTOR_DIR + 'AD_in_the_family_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_ad_ip
group2 = [TAU_CAT, AD_IF]

stat_test(data_type, group1, group2) # Too few samples for the test

### Herpes_zoster_vaccine

In [ ]:
HZV = 'Herpes_zoster_vaccine'

df_high_vs_low_hzv = df_high_vs_low.dropna(subset=[HZV])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_hzv, HZV, TAU_CAT, 2, grpah_colors, 'Herpes zoster virus vs Tau 217 category', RISK_FACTOR_DIR + 'Herpes_zoster_virus_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hzv
group2 = [TAU_CAT, HZV]

stat_test(data_type, group1, group2) # Too few samples for the test

## Blood Tests

In [ ]:
BLOOD_TESTS_SAVE_DIR = SAVE_DIR + 'blood_tests/'

### Hemoglobin_A1C

In [ ]:
HA1C = 'Hemoglobin_A1C'

df_high_vs_low_ha1c = df_high_vs_low.dropna(subset=[HA1C])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ha1c, HA1C, TAU_CAT, graph_colors, 'Hemoglobin A1C vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Hemoglobin_A1C_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Catagorical data
HA1C_INT = 5.7
HA1C_HIGH = 6.4

df_high_vs_low_ha1c['HA1C_CAT'] = pd.cut(df_high_vs_low_ha1c[HA1C], bins=[0, HA1C_INT, HA1C_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ha1c, 'HA1C_CAT', TAU_CAT, 3, grpah_colors, 'Hemoglobin A1C vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Hemoglobin_A1C_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'Low'][HA1C]
group2 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'High'][HA1C]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'Low']['HA1C_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'High']['HA1C_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

stat_test(data_type, group1, group2)

### Cholesterol

In [ ]:
CHOL = 'Cholesterol'

df_high_vs_low_chol = df_high_vs_low.dropna(subset=[CHOL])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_chol, CHOL, TAU_CAT, graph_colors, 'Cholesterol vs Tau 217 category', SAVE_DIR + 'Cholesterol_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Categorical data
CHOL_INT = 180
CHOL_HIGH = 240

df_high_vs_low_chol['CHOL_CAT'] = pd.cut(df_high_vs_low_chol[CHOL], bins=[0, CHOL_INT, CHOL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_chol, 'CHOL_CAT', TAU_CAT, 3, grpah_colors, 'Cholesterol vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Cholesterol_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'Low'][CHOL]
group2 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'High'][CHOL]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'Low']['CHOL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'High']['CHOL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

stat_test(data_type, group1, group2)

### Triglycerides

In [ ]:
TRIG = 'Triglycerides'

df_high_vs_low_trig = df_high_vs_low.dropna(subset=[TRIG])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_trig, TRIG, TAU_CAT, graph_colors, 'Triglycerides vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Triglycerides_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Catagorical Data
TRG_INT = 150
TRG_HIGH = 200

df_high_vs_low_trig['TRG_CAT'] = pd.cut(df_high_vs_low_trig[TRIG], bins=[0, TRG_INT, TRG_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_trig, 'TRG_CAT', TAU_CAT, 3, grpah_colors, 'Triglycerides vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Triglycerides_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'Low'][TRIG].astype(float)
group2 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'High'][TRIG].astype(float)

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'Low']['TRG_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'High']['TRG_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

stat_test(data_type, group1, group2)

### LDL

In [ ]:
LDL = 'LDL'

df_high_vs_low_ldl = df_high_vs_low.dropna(subset=[LDL])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ldl, LDL, TAU_CAT, graph_colors, 'LDL vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'LDL_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
LDL_INT = 100
LDL_HIGH = 130

df_high_vs_low_ldl['LDL_CAT'] = pd.cut(df_high_vs_low_ldl[LDL], bins=[0, LDL_INT, LDL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ldl, 'LDL_CAT', TAU_CAT, 3, grpah_colors, 'LDL vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'LDL_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'Low'][LDL]
group2 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'High'][LDL]

stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'Low']['LDL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'High']['LDL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

stat_test(data_type, group1, group2)

### HDL

In [ ]:
HDL = 'HDL'

df_high_vs_low_hdl = df_high_vs_low.dropna(subset=[HDL])

kdeplot_high_vs_low(df_high_vs_low_hdl, HDL, TAU_CAT, graph_colors, 'HDL vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'HDL_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_hdl[df_high_vs_low_hdl[TAU_CAT] == 'Low'][HDL]
group2 = df_high_vs_low_hdl[df_high_vs_low_hdl[TAU_CAT] == 'High'][HDL]

stat_test(data_type, group1, group2)

### CRP

In [ ]:
df_high_vs_low['CRP'].replace({'<0.5': 0, '<0.6':0}, inplace=True)

In [ ]:
# Cataogrical data
CRP = 'CRP'
CRP_INT = 0.5
CRP_HIGH = 1
CRP_VERY_HIGH = 2
# df_high_vs_low['CRP'] = df_high_vs_low['CRP'].astype(float)
df_high_vs_low_crp = df_high_vs_low.dropna(subset=[CRP])
df_high_vs_low_crp['CRP_CAT'] = pd.cut(df_high_vs_low_crp[CRP], bins=[0, CRP_INT, CRP_HIGH, CRP_VERY_HIGH, np.inf], labels=['Low', 'Intermediate', 'High', 'Very High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_crp, 'CRP_CAT', TAU_CAT, 4, grpah_colors, 'CRP vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'CRP_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low'][CRP]
group2 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'High'][CRP]

stat_test(data_type, group1, group2)

In [ ]:
df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).unique()
df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].unique()

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).dropna()
group2 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'High']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).dropna()

stat_test(data_type, group1, group2)

### HB

In [ ]:
HB = 'HB'

df_high_vs_low_hb = df_high_vs_low.dropna(subset=[HB])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_hb, HB, TAU_CAT, graph_colors, 'HB vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'HB_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_hb[df_high_vs_low_hb[TAU_CAT] == 'Low'][HB]
group2 = df_high_vs_low_hb[df_high_vs_low_hb[TAU_CAT] == 'High'][HB]

stat_test(data_type, group1, group2)

### RBC

In [ ]:
RBC = 'RBC'

df_high_vs_low_rbc = df_high_vs_low.dropna(subset=[RBC])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_rbc, RBC, TAU_CAT, graph_colors, 'RBC vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'RBC_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_rbc[df_high_vs_low_rbc[TAU_CAT] == 'Low'][RBC]
group2 = df_high_vs_low_rbc[df_high_vs_low_rbc[TAU_CAT] == 'High'][RBC]

stat_test(data_type, group1, group2)

### WBC

In [ ]:
WBC = 'WBC'

df_high_vs_low_wbc = df_high_vs_low.dropna(subset=[WBC])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_wbc, WBC, TAU_CAT, graph_colors, 'WBC vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'WBC_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
WBC_LOW = 4.5
WBC_HIGH = 11

df_high_vs_low_wbc['WBC_CAT'] = pd.cut(df_high_vs_low_wbc[WBC], bins=[0, WBC_LOW, WBC_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_wbc, 'WBC_CAT', TAU_CAT, 3, grpah_colors, 'WBC vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'WBC_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_wbc[df_high_vs_low_wbc[TAU_CAT] == 'Low'][WBC]
group2 = df_high_vs_low_wbc[df_high_vs_low_wbc[TAU_CAT] == 'High'][WBC]

stat_test(data_type, group1, group2)

### Lymphocytes

In [ ]:
LYMPH = 'Lymphocytes'

df_high_vs_low_lymph = df_high_vs_low.dropna(subset=[LYMPH])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_lymph, LYMPH, TAU_CAT, graph_colors, 'Lymphocytes vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Lymphocytes_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
LYMPH_LOW = 1.3
LYMPH_HIGH = 4.7

df_high_vs_low_lymph['LYMPH_CAT'] = pd.cut(df_high_vs_low_lymph[LYMPH], bins=[0, LYMPH_LOW, LYMPH_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_lymph, 'LYMPH_CAT', TAU_CAT, 3, grpah_colors, 'Lymphocytes vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Lymphocytes_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_lymph[df_high_vs_low_lymph[TAU_CAT] == 'Low'][LYMPH]
group2 = df_high_vs_low_lymph[df_high_vs_low_lymph[TAU_CAT] == 'High'][LYMPH]

stat_test(data_type, group1, group2)

In [ ]:
group1['LYMPH_CAT']

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_lymph[df_high_vs_low_lymph['LYMPH_CAT'] != 'High'] # Theres only two subjects with 'High' value
group1['LYMPH_CAT_num'] = group1['LYMPH_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2}).astype(int)
group2 = [TAU_CAT, 'LYMPH_CAT_num']

stat_test(data_type, group1, group2)

### Creatinine

In [ ]:
CREATININE = 'Creatinine'

df_high_vs_low_creatinine = df_high_vs_low.dropna(subset=[CREATININE])
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_creatinine, CREATININE, TAU_CAT, graph_colors, 'Creatinine vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Creatinine_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_creatinine[df_high_vs_low_creatinine[TAU_CAT] == 'Low'][CREATININE]
group2 = df_high_vs_low_creatinine[df_high_vs_low_creatinine[TAU_CAT] == 'High'][CREATININE]

stat_test(data_type, group1, group2)